# Exploration

```{important} AI Disclosure
The written content in this report was drafted by Jenny Lee and refined using Generative AI for clarity and grammar. All ideas, analysis decisions, and interpretations are original. The prompt used throughout was: *"Refine my words."*
```

```{note} Viewing the Context
The written content in this report was drafted by Jenny Lee and refined using Generative AI for clarity and grammar. All ideas, analysis decisions, and interpretations are original. The prompt used throughout was: *"Refine my words."*
```

## Data Collection

🔗 [Data Source](https://imdbapi.dev)

In [26]:
import pandas as pd
import numpy as np
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import json
import ast

## E1
> If you dataset has an unusual structure, describe the steps and
related challenges in extracting the dataset. For example, if your data comes from an API, talk about the
API calls you had to run in order to get the data.

In [17]:
BASE_URL = "https://api.imdbapi.dev/"
response = requests.get(f"{BASE_URL}/titles?types=MOVIE&startYear>=2024")
data = response.json()
df = pd.DataFrame(data["titles"])
pagetoken = data["nextPageToken"]
time.sleep(1)  

for page in range(1, 20):
      try:
            response = requests.get(f"{BASE_URL}/titles?types=MOVIE&startYear=2026&pageToken={pagetoken}", timeout=10)
            
            if response.status_code == 429:
                  wait_time = int(response.headers.get("Retry-After", 5))
                  print(f"Page {page}: Rate limited! Waiting {wait_time}s...")
                  time.sleep(wait_time)
                  continue
            
            response.raise_for_status()
            
            if not response.text:
                  break
            
            data = response.json()
            
            if "titles" not in data or not data["titles"]:
                  break
                  
            df_page = pd.DataFrame(data["titles"])
            df = pd.concat([df, df_page], ignore_index=True)
            
            if "nextPageToken" in data and data["nextPageToken"]:
                  pagetoken = data["nextPageToken"]
            else:
                  break
            
            time.sleep(1)  
      except Exception as e:
            break

print(f"Final shape: {df.shape}")
print(f"Total Number of rows: {len(df)}")
display(df.head(10))

Final shape: (1000, 10)
Total Number of rows: 1000


,id,type,primaryTitle,originalTitle,primaryImage,startYear,runtimeSeconds,genres,rating,plot
0,tt12042730,movie,Project Hail Mary,Project Hail Mary,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,9360.0,"[Adventure, Comedy, Sci-Fi]","{'aggregateRating': 8.4, 'voteCount': 178625}",A science teacher wakes up alone on a spaceshi...
1,tt28650488,movie,The Super Mario Galaxy Movie,The Super Mario Galaxy Movie,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,5880.0,"[Animation, Adventure, Comedy, Family, Fantasy]","{'aggregateRating': 6.5, 'voteCount': 25522}","Mario ventures into space, exploring cosmic wo..."
2,tt32430579,movie,Crime 101,Crime 101,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,8400.0,"[Crime, Drama, Thriller]","{'aggregateRating': 6.8, 'voteCount': 48321}","An elusive thief, eyeing his final score, enco..."
3,tt33071426,movie,The Drama,The Drama,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,6300.0,"[Comedy, Drama, Romance]","{'aggregateRating': 7.5, 'voteCount': 17072}",A happily engaged couple is put to the test wh...
4,tt37209937,movie,Pizza Movie,Pizza Movie,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,5820.0,[Comedy],"{'aggregateRating': 5.8, 'voteCount': 4258}",High college students face an unexpectedly epi...
5,tt27543632,movie,The Housemaid,The Housemaid,{'url': 'https://m.media-amazon.com/images/M/M...,2025.0,7860.0,"[Drama, Mystery, Thriller]","{'aggregateRating': 6.8, 'voteCount': 128853}",A struggling young woman is relieved by the ch...
6,tt33244668,movie,Anaconda,Anaconda,{'url': 'https://m.media-amazon.com/images/M/M...,2025.0,5940.0,"[Action, Adventure, Comedy]","{'aggregateRating': 5.6, 'voteCount': 52429}",A group of friends are going through a mid-lif...
7,tt27552099,movie,Mike & Nick & Nick & Alice,Mike & Nick & Nick & Alice,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,6420.0,"[Action, Comedy, Crime]","{'aggregateRating': 6.2, 'voteCount': 12231}",Two friends navigate the dangerous world of or...
8,tt39139925,movie,Dhurandhar The Revenge,Dhurandhar The Revenge,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,13800.0,"[Action, Crime, Thriller]","{'aggregateRating': 8.5, 'voteCount': 43693}",Jaskirat Singh Rangi descends deeper into his ...
9,tt0049833,movie,The Ten Commandments,The Ten Commandments,{'url': 'https://m.media-amazon.com/images/M/M...,1956.0,13200.0,"[Adventure, Drama, Family, History]","{'aggregateRating': 7.9, 'voteCount': 84370}","Moses, raised as a prince of Egypt in the Phar..."


Data was collected across multiple endpoints for various purposes outlined throughout this report, with the `/titles` endpoint serving as the main source for general movie metadata. Results were paginated at 50 entries per page, with each response returning a token to access the next page; this pagination mechanism was used to retrieve a total of 1,000 rows filtered by `genre: MOVIE`.

The primary challenge in data extraction was managing the API's rate limits. To mitigate this, deliberate delays were introduced in the retrieval code, by pausing for 1 second between requests and extending to 5 seconds upon failure, ensuring stable and uninterrupted data collection.

```{code-cell} python
:linenos:

def fetch_box_office(ind, row, base_url, delay=0.5):
    title_id = row["id"]
    time.sleep(delay)  
    
    try:
        response = requests.get(f"{base_url}/titles/{title_id}/boxOffice", timeout=5)
        
        if response.status_code == 429:
            wait_time = int(response.headers.get("Retry-After", 10))
            print(f"Rate limited on {title_id}, waiting {wait_time}s...")
            time.sleep(wait_time)
            return fetch_box_office(ind, row, base_url, delay=2)
        
        response.raise_for_status() 
        data = response.json()
        worldwide_gross = data["worldwideGross"]["amount"] if "worldwideGross" in data else None
        worldwide_gross_currency = data["worldwideGross"]["currency"] if "worldwideGross" in data else None
        production_budget = data["productionBudget"]["amount"] if "productionBudget" in data else None
        production_budget_currency = data["productionBudget"]["currency"] if "productionBudget" in data else None

        return ind, worldwide_gross, worldwide_gross_currency, production_budget, production_budget_currency, None, data 

skipped_count = 0
printed_sample = False

for ind, row in df.iterrows():
    ind_result, worldwide_gross, worldwide_gross_currency, production_budget, production_budget_currency, error, data = fetch_box_office(ind, row, BASE_URL)

    if not printed_sample and not error and data:
        print("Sample API response to show nested data:")
        print(data)
        printed_sample = True

    if error:
        skipped_count += 1
    else:
        df.at[ind, "worldwideGross"] = worldwide_gross
        df.at[ind, "worldwideGrossCurrency"] = worldwide_gross_currency
        df.at[ind, "productionBudget"] = production_budget
        df.at[ind, "productionBudgetCurrency"] = production_budget_currency

print(f"Skipped {skipped_count} titles due to errors.")
print(f"Box office data added to {len(df) - skipped_count} rows.")

display(df.head(10))

```

In [ ]:
def fetch_box_office(ind, row, base_url, delay=0.5):
    title_id = row["id"]
    time.sleep(delay)  
    try:
        response = requests.get(f"{base_url}/titles/{title_id}/boxOffice", timeout=5)
        
        if response.status_code == 429:
            wait_time = int(response.headers.get("Retry-After", 10))
            print(f"Rate limited on {title_id}, waiting {wait_time}s...")
            time.sleep(wait_time)
            return fetch_box_office(ind, row, base_url, delay=2)
        
        response.raise_for_status() 
        data = response.json()
        worldwide_gross = data["worldwideGross"]["amount"] if "worldwideGross" in data else None
        worldwide_gross_currency = data["worldwideGross"]["currency"] if "worldwideGross" in data else None
        production_budget = data["productionBudget"]["amount"] if "productionBudget" in data else None
        production_budget_currency = data["productionBudget"]["currency"] if "productionBudget" in data else None

        return ind, worldwide_gross, worldwide_gross_currency, production_budget, production_budget_currency, None, data 
    except requests.Timeout:
        return ind, None, None, None, None, f"{title_id}: Timeout", None
    except requests.HTTPError as e:
        return ind, None, None, None, None, f"{title_id}: HTTP {response.status_code}", None
    except json.JSONDecodeError:
        return ind, None, None, None, None, f"{title_id}: Invalid JSON response", None
    except Exception as e:
        return ind, None, None, None, None, f"{title_id}: {type(e).__name__}", None

skipped_count = 0
printed_sample = False

for ind, row in df.iterrows():
    ind_result, worldwide_gross, worldwide_gross_currency, production_budget, production_budget_currency, error, data = fetch_box_office(ind, row, BASE_URL)

    if not printed_sample and not error and data:
        print("Sample API response to show nested data:")
        print(data)
        printed_sample = True

    if error:
        skipped_count += 1
    else:
        df.at[ind, "worldwideGross"] = worldwide_gross
        df.at[ind, "worldwideGrossCurrency"] = worldwide_gross_currency
        df.at[ind, "productionBudget"] = production_budget
        df.at[ind, "productionBudgetCurrency"] = production_budget_currency

display(df.head(5))

Sample API response to show nested data:
{'domesticGross': {'amount': '229071756', 'currency': 'USD'}, 'worldwideGross': {'amount': '433030505', 'currency': 'USD'}, 'openingWeekendGross': {'gross': {'amount': '80506007', 'currency': 'USD'}}, 'productionBudget': {'amount': '200000000', 'currency': 'USD'}}
Skipped 0 titles due to errors.
Box office data added to 1000 rows.


,id,type,primaryTitle,originalTitle,primaryImage,startYear,runtimeSeconds,genres,rating,plot,worldwideGross,worldwideGrossCurrency,productionBudget,productionBudgetCurrency
0,tt12042730,movie,Project Hail Mary,Project Hail Mary,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,9360.0,"[Adventure, Comedy, Sci-Fi]","{'aggregateRating': 8.4, 'voteCount': 178625}",A science teacher wakes up alone on a spaceshi...,433030505,USD,200000000,USD
1,tt28650488,movie,The Super Mario Galaxy Movie,The Super Mario Galaxy Movie,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,5880.0,"[Animation, Adventure, Comedy, Family, Fantasy]","{'aggregateRating': 6.5, 'voteCount': 25522}","Mario ventures into space, exploring cosmic wo...",437751829,USD,110000000,USD
2,tt32430579,movie,Crime 101,Crime 101,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,8400.0,"[Crime, Drama, Thriller]","{'aggregateRating': 6.8, 'voteCount': 48321}","An elusive thief, eyeing his final score, enco...",72559167,USD,90000000,USD
3,tt33071426,movie,The Drama,The Drama,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,6300.0,"[Comedy, Drama, Romance]","{'aggregateRating': 7.5, 'voteCount': 17072}",A happily engaged couple is put to the test wh...,26232083,USD,None,None
4,tt37209937,movie,Pizza Movie,Pizza Movie,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,5820.0,[Comedy],"{'aggregateRating': 5.8, 'voteCount': 4258}",High college students face an unexpectedly epi...,None,None,None,None
5,tt27543632,movie,The Housemaid,The Housemaid,{'url': 'https://m.media-amazon.com/images/M/M...,2025.0,7860.0,"[Drama, Mystery, Thriller]","{'aggregateRating': 6.8, 'voteCount': 128853}",A struggling young woman is relieved by the ch...,399318859,USD,35000000,USD
6,tt33244668,movie,Anaconda,Anaconda,{'url': 'https://m.media-amazon.com/images/M/M...,2025.0,5940.0,"[Action, Adventure, Comedy]","{'aggregateRating': 5.6, 'voteCount': 52429}",A group of friends are going through a mid-lif...,134974943,USD,45000000,USD
7,tt27552099,movie,Mike & Nick & Nick & Alice,Mike & Nick & Nick & Alice,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,6420.0,"[Action, Comedy, Crime]","{'aggregateRating': 6.2, 'voteCount': 12231}",Two friends navigate the dangerous world of or...,None,None,None,None
8,tt39139925,movie,Dhurandhar The Revenge,Dhurandhar The Revenge,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,13800.0,"[Action, Crime, Thriller]","{'aggregateRating': 8.5, 'voteCount': 43693}",Jaskirat Singh Rangi descends deeper into his ...,150917485,USD,None,None
9,tt0049833,movie,The Ten Commandments,The Ten Commandments,{'url': 'https://m.media-amazon.com/images/M/M...,1956.0,13200.0,"[Adventure, Drama, Family, History]","{'aggregateRating': 7.9, 'voteCount': 84370}","Moses, raised as a prince of Egypt in the Phar...",65501273,USD,13282712,USD


The primary goal of this project is to identify key movie characteristics that drive a high worldwide gross relative to production budget, and to leverage those characteristics to build a predictive model for the gross-to-budget ratio. To retrieve the necessary financial data, the `/titles/{titleId}/boxOffice` endpoint was used, where `titleId` corresponds to the `id` column in the dataset.

## E2
> If you dataset has an unusual structure, describe the steps and related challenges in transforming and loading the dataset. For example, if your dataset is too large to
load all at once, talk about batch loading and/or the bigmemory package. If your dataset is images, talk
about file formatting issues and how you look at individual pixels. If your dataset is nested data frames,
talk about how you flatten and simplify that that.

### Nested Data
As shown in the code above (lines 16–19), the data retrieved from `/titles/{titleId}/boxOffice` is structured as a nested dictionary. In particular, both `worldwideGross` and `productionBudget` contain subfields, requiring separate extraction of the amount and currency values.

Now, I would like to incorporate an additional feature from the `chart/starmeter` endpoint. This metric provides a weekly ranking that reflects the popularity of actors, directors, and other industry professionals based on their page views on the IMDb website. The goal is to examine whether the presence of more popular actors has an impact on the gross-to-budget ratio.

In [ ]:
def check_starmeter(base_url=BASE_URL, max_pages=100, delay=1.0):
      response = requests.get(f"{base_url}/chart/starmeter", timeout=10)
      response.raise_for_status()
      data = response.json()
      items = []
      names = data.get("names", [])
      nextPageToken = data.get("nextPageToken")
      items.extend([{**name, "page": 0} for name in names])
      
      for page in range(1, max_pages):
            if not nextPageToken:
                  break
            response = requests.get(f"{base_url}/chart/starmeter?pageToken={nextPageToken}", timeout=10)
            if response.status_code == 429:
                  wait_time = int(response.headers.get("Retry-After", 5))
                  time.sleep(wait_time)
                  continue
            
            response.raise_for_status()
            data = response.json()
            names = data.get("names", [])
            nextPageToken = data.get("nextPageToken")
            if not names:
                  break
            items.extend([{**name, "page": page} for name in names])
            time.sleep(delay)
      
      return pd.DataFrame(items)

star_df = check_starmeter()
display(star_df.head(5))

,id,displayName,primaryImage,heightCm,birthDate,meterRanking,page,deathDate
0,nm6714979,Camila Morrone,{'url': 'https://m.media-amazon.com/images/M/M...,175.0,"{'year': 1997, 'month': 6, 'day': 16}","{'currentRank': 1, 'changeDirection': 'UP', 'd...",0,NaN
1,nm1172478,Joel Kinnaman,{'url': 'https://m.media-amazon.com/images/M/M...,190.0,"{'year': 1979, 'month': 11, 'day': 25}","{'currentRank': 2, 'changeDirection': 'UP', 'd...",0,NaN
2,nm0331516,Ryan Gosling,{'url': 'https://m.media-amazon.com/images/M/M...,184.0,"{'year': 1980, 'month': 11, 'day': 12}","{'currentRank': 3, 'changeDirection': 'DOWN', ...",0,NaN
3,nm0000492,Jennifer Jason Leigh,{'url': 'https://m.media-amazon.com/images/M/M...,160.0,"{'year': 1962, 'month': 2, 'day': 5}","{'currentRank': 4, 'changeDirection': 'UP', 'd...",0,NaN
4,nm1197689,Sandra Hüller,{'url': 'https://m.media-amazon.com/images/M/M...,173.0,"{'year': 1978, 'month': 4, 'day': 30}","{'currentRank': 5, 'changeDirection': 'DOWN', ...",0,NaN
5,nm4422686,Barry Keoghan,{'url': 'https://m.media-amazon.com/images/M/M...,173.0,"{'year': 1992, 'month': 10, 'day': 18}","{'currentRank': 6, 'changeDirection': 'DOWN', ...",0,NaN
6,nm1683768,Beau Garrett,{'url': 'https://m.media-amazon.com/images/M/M...,178.0,"{'year': 1982, 'month': 12, 'day': 28}","{'currentRank': 7, 'changeDirection': 'DOWN', ...",0,NaN
7,nm6262342,Gus Birney,{'url': 'https://m.media-amazon.com/images/M/M...,175.0,"{'year': 1999, 'month': 7, 'day': 27}","{'currentRank': 8, 'changeDirection': 'UP', 'd...",0,NaN
8,nm5954280,Nicholas Galitzine,{'url': 'https://m.media-amazon.com/images/M/M...,183.0,"{'year': 1994, 'month': 9, 'day': 29}","{'currentRank': 9, 'changeDirection': 'UP', 'd...",0,NaN
9,nm2858875,Sydney Sweeney,{'url': 'https://m.media-amazon.com/images/M/M...,161.0,"{'year': 1997, 'month': 9, 'day': 12}","{'currentRank': 10, 'changeDirection': 'UP', '...",0,NaN


In [27]:
star_df["currentRank"] = star_df["meterRanking"].apply(lambda x: ast.literal_eval(x).get("currentRank") if isinstance(x, str) else (x.get("currentRank") if isinstance(x, dict) else None))
star_df

,id,displayName,primaryImage,heightCm,birthDate,meterRanking,page,deathDate,currentRank
0,nm6714979,Camila Morrone,{'url': 'https://m.media-amazon.com/images/M/M...,175.0,"{'year': 1997, 'month': 6, 'day': 16}","{'currentRank': 1, 'changeDirection': 'UP', 'd...",0,NaN,1.0
1,nm1172478,Joel Kinnaman,{'url': 'https://m.media-amazon.com/images/M/M...,190.0,"{'year': 1979, 'month': 11, 'day': 25}","{'currentRank': 2, 'changeDirection': 'UP', 'd...",0,NaN,2.0
2,nm0331516,Ryan Gosling,{'url': 'https://m.media-amazon.com/images/M/M...,184.0,"{'year': 1980, 'month': 11, 'day': 12}","{'currentRank': 3, 'changeDirection': 'DOWN', ...",0,NaN,3.0
3,nm0000492,Jennifer Jason Leigh,{'url': 'https://m.media-amazon.com/images/M/M...,160.0,"{'year': 1962, 'month': 2, 'day': 5}","{'currentRank': 4, 'changeDirection': 'UP', 'd...",0,NaN,4.0
4,nm1197689,Sandra Hüller,{'url': 'https://m.media-amazon.com/images/M/M...,173.0,"{'year': 1978, 'month': 4, 'day': 30}","{'currentRank': 5, 'changeDirection': 'DOWN', ...",0,NaN,5.0
...,...,...,...,...,...,...,...,...,...
9995,nm16540376,Shanna Sinow,{'url': 'https://m.media-amazon.com/images/M/M...,163.0,NaN,NaN,99,NaN,NaN
9996,nm12168055,Adrian Greensmith,{'url': 'https://m.media-amazon.com/images/M/M...,185.0,"{'year': 2001, 'month': 12, 'day': 23}",NaN,99,NaN,NaN
9997,nm6255947,Jack Rafferty,{'url': 'https://m.media-amazon.com/images/M/M...,178.0,NaN,NaN,99,NaN,NaN
9998,nm10169245,Sam Houston,NaN,NaN,NaN,NaN,99,NaN,NaN


### Data Transformation 

As before, the nested `currentRank` field within `meterRanking` is extracted into a separate column. Rather than using the raw rank values, the rankings are grouped into tiers of 100. This transformation reduces noise from small, insignificant differences between adjacent ranks and better captures the nonlinear nature of popularity, where top-ranked individuals differ much more meaningfully from lower-ranked ones. Given that there are approximately 5,000 ranks, values beyond this range are assigned to a baseline tier of 0. The highest tier, corresponding to ranks under 100, is assigned a value of 10.

The second dataframe below displays the distribution of observations across the tiers.

In [ ]:
def get_tier(rank):
    try:
        if isinstance(rank, (pd.Series, list, dict)):
            return None
        if pd.isna(rank):
            return None

        rank_val = int(float(rank))

        if rank_val < 1 or rank_val > 1000:
            return 0  

        return 11 - ((rank_val - 1) // 100 + 1)

    except (TypeError, ValueError):
        return None

star_df["currentRank_Tier"] = star_df["currentRank"].apply(get_tier)
display(star_df[["id", "currentRank_Tier"]].head(5))
display(star_df["currentRank_Tier"].value_counts())

,id,currentRank_Tier
0,nm6714979,10.0
1,nm1172478,10.0
2,nm0331516,10.0
3,nm0000492,10.0
4,nm1197689,10.0
5,nm4422686,10.0
6,nm1683768,10.0
7,nm6262342,10.0
8,nm5954280,10.0
9,nm2858875,10.0


currentRank_Tier
0.0     3993
10.0     100
9.0      100
8.0      100
7.0      100
6.0      100
5.0      100
4.0      100
3.0      100
1.0      100
2.0       99
Name: count, dtype: int64

In [33]:
df["totalStarMeter"] = 0

for ind, row in df.iterrows():
    retries = 3
    while retries > 0:
        try:
            response = requests.get(f"{BASE_URL}/titles/{row['id']}")
            
            if response.status_code == 429:
                time.sleep(10)
                retries -= 1
                continue
            
            if response.status_code != 200 or not response.text.strip():
                break

            data = response.json()
            stars = data.get("stars", [])

            total = 0
            for star in stars:
                star_id = star["id"]
                match = star_df[star_df["id"] == star_id]
                if not match.empty:
                    tier = match.iloc[0]["currentRank_Tier"]
                    total += int(tier) if pd.notna(tier) else 0
            
            df.at[ind, "totalStarMeter"] = total
            time.sleep(0.5) 
            break 

        except Exception as e:
            break
    else:
        print(f"Row {ind}: Skipped after 3 retries")

df[["id", "totalStarMeter"]].head(5)

,id,totalStarMeter
0,tt12042730,37
1,tt28650488,66
2,tt32430579,49
3,tt33071426,27
4,tt37209937,25


In [35]:
df[df["totalStarMeter"] == df["totalStarMeter"].max()]

,id,type,primaryTitle,originalTitle,primaryImage,startYear,runtimeSeconds,genres,rating,plot,worldwideGross,worldwideGrossCurrency,productionBudget,productionBudgetCurrency,totalStarMeter
1,tt28650488,movie,The Super Mario Galaxy Movie,The Super Mario Galaxy Movie,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,5880.0,"[Animation, Adventure, Comedy, Family, Fantasy]","{'aggregateRating': 6.5, 'voteCount': 25522}","Mario ventures into space, exploring cosmic wo...",437751829,USD,110000000,USD,66


Now, a new feature, `totalStarMeter`, has been introduced for each movie, representing the sum of the tiered ratings of all associated cast members. By the end of the code above, the two dataframes, sourced from different API endpoints, have been successfully merged into a single dataset. The movie with the highest `totalStarMeter` is **The Super Mario Galaxy** in 2026. 

In [37]:
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")

Total rows: 1000
Total columns: 15


In [39]:
df.to_csv("../data/box_office_data_with_starmeter.csv", index=False)